In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Brute Force\\combined_brute_force.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "brute_force_cnn.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Brute Force\combined_brute_force.csv | Total rows: 7238
Train: 4342 | Val: 1448 | Test: 1448


In [3]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train.values.copy())
X_val_t   = torch.FloatTensor(X_val_scaled)
y_val_t   = torch.FloatTensor(y_val.values.copy())
X_test_t  = torch.FloatTensor(X_test_scaled)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear(128 * (input_dim // 4), 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x.unsqueeze(1)).squeeze(1)

model     = CNNModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

In [9]:
epochs = 50
patience = 5
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t.to(device))
        val_loss = criterion(val_preds, y_val_t.to(device)).item()
        val_acc   = ((val_preds > 0.5) == y_val_t.to(device)).float().mean().item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# Load best model before evaluation
model.load_state_dict(torch.load(MODEL_PATH))

Epoch 1/50 | Loss: 0.6360 | Val Loss: 0.5858 | Val Acc: 0.6968
Epoch 2/50 | Loss: 0.5467 | Val Loss: 0.5240 | Val Acc: 0.6920
Epoch 3/50 | Loss: 0.5092 | Val Loss: 0.4958 | Val Acc: 0.7134
Epoch 4/50 | Loss: 0.4928 | Val Loss: 0.4844 | Val Acc: 0.7238
Epoch 5/50 | Loss: 0.4808 | Val Loss: 0.4798 | Val Acc: 0.7514
Epoch 6/50 | Loss: 0.4724 | Val Loss: 0.4650 | Val Acc: 0.7762
Epoch 7/50 | Loss: 0.4672 | Val Loss: 0.4803 | Val Acc: 0.7604
Epoch 8/50 | Loss: 0.4610 | Val Loss: 0.4584 | Val Acc: 0.7894
Epoch 9/50 | Loss: 0.4518 | Val Loss: 0.4519 | Val Acc: 0.7728
Epoch 10/50 | Loss: 0.4453 | Val Loss: 0.4528 | Val Acc: 0.7859
Epoch 11/50 | Loss: 0.4394 | Val Loss: 0.4400 | Val Acc: 0.7749
Epoch 12/50 | Loss: 0.4356 | Val Loss: 0.4398 | Val Acc: 0.7728
Epoch 13/50 | Loss: 0.4293 | Val Loss: 0.4304 | Val Acc: 0.8004
Epoch 14/50 | Loss: 0.4238 | Val Loss: 0.4251 | Val Acc: 0.8025
Epoch 15/50 | Loss: 0.4237 | Val Loss: 0.4405 | Val Acc: 0.7790
Epoch 16/50 | Loss: 0.4209 | Val Loss: 0.4250 | V

<All keys matched successfully>

In [ ]:
model.eval()
with torch.no_grad():
    preds = (model(X_test_t.to(device)) > 0.5).cpu().numpy().astype(int)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.88      0.75      0.81       724
      Attack       0.78      0.90      0.84       724

    accuracy                           0.83      1448
   macro avg       0.83      0.83      0.82      1448
weighted avg       0.83      0.83      0.82      1448

[[546 178]
 [ 75 649]]


: 